In [27]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [28]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()

    con.executescript(script)
    con.commit()

    print("Run:", path)

In [29]:
run_sql_file(r"C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\01_revenue_customer.sql")

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\01_revenue_customer.sql


In [30]:
q("""select
count(*)as total_customers,
round(avg(arpu_inr),2)as average_arpu,
count(distinct arpu_inr)as distinct_arpu_values
from customers;
""")

,total_customers,average_arpu,distinct_arpu_values
0,22000,421.65,13


In [31]:
q("""SELECT
COUNT(*) AS customers_without_payments
FROM customers AS c
WHERE NOT EXISTS (
SELECT 1
FROM recharges AS r
WHERE r.customer_id = c.customer_id
AND r.recharge_date >= '2025-01-01'
AND r.recharge_date < '2025-07-01');""")

,customers_without_payments
0,3711


In [32]:
q("""
select name,type
from sqlite_master
where name = 'v_revenue_customer';
""")

,name,type
0,v_revenue_customer,view


In [33]:
q("""
select * from v_revenue_customer;
""")

,customer_id,recharge_count,recharge_total_6m,actual_arpu_6m
0,1,3,897,149.50
1,2,2,198,33.00
2,3,6,5073,845.50
3,4,0,0,0.00
4,5,2,1438,239.67
...,...,...,...,...
21995,21996,6,10478,1746.33
21996,21997,1,719,119.83
21997,21998,5,1045,174.17
21998,21999,2,798,133.00


In [34]:
q("""SELECT
SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END) AS customers_with_payments,
SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) AS customers_without_payments
FROM v_revenue_customer;""")

,customers_with_payments,customers_without_payments
0,18289,3711


In [35]:
run_sql_file(r"C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\02_usage_customer.sql")

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\02_usage_customer.sql


In [36]:
q("""select * from v_usage_customer;"""
  )

,customer_id,usage_days,total_data_mb,total_voice_min,total_sms,roaming_days,mb_per_active_day,mb_per_observed_day,last_usage_date,recency_days
0,1,17,12405.0,333,3,0,729.71,68.54,2025-04-26,65.0
1,2,11,2822.5,424,6,1,256.59,15.59,2025-02-11,139.0
2,3,55,102791.3,2205,23,1,1868.93,567.91,2025-06-25,5.0
3,4,40,8755.2,2262,26,0,218.88,48.37,2025-03-16,106.0
4,5,39,44725.0,1364,9,0,1146.79,247.10,2025-06-18,12.0
...,...,...,...,...,...,...,...,...,...,...
21995,21996,79,522264.4,4713,51,2,6610.94,2885.44,2025-06-30,0.0
21996,21997,101,141403.8,6621,83,2,1400.04,781.24,2025-06-28,2.0
21997,21998,65,4881.9,2111,30,6,75.11,26.97,2025-06-30,0.0
21998,21999,18,20771.6,596,11,0,1153.98,114.76,2025-03-31,91.0


In [37]:
q("""select count(*) as customers_with_complaints 
from customers as c 
left join complaints as cp on c.customer_id = cp.customer_id 
where cp.customer_id is null;""")

,customers_with_complaints
0,6631


In [38]:
run_sql_file(r'C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\03_complaint_customer.sql')

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\03_complaint_customer.sql


In [39]:
q("""select count(*) from v_complaint_customer;""")

,count(*)
0,22000


In [40]:
q("""select * from v_complaint_customer;""")

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,1,1,1,0,NaN
1,2,1,1,0,NaN
2,3,0,0,0,NaN
3,4,0,0,0,NaN
4,5,2,1,1,13.0
...,...,...,...,...,...
21995,21996,2,1,1,11.0
21996,21997,1,0,1,13.0
21997,21998,1,0,1,10.0
21998,21999,0,0,0,NaN


In [41]:
q("""select count(*) from v_complaint_customer where complaint_count = 0;""")

,count(*)
0,6631


In [42]:
run_sql_file(r'C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\04_plan_clean.sql')

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\04_plan_clean.sql


In [43]:
q("""select * from v_plans_clean;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,quota_basis,quota_gb,monthly_gb,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,per day,0.5,15.0,NaN,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,per day,1.0,30.0,NaN,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,per day,1.0,30.0,NaN,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,per day,1.5,45.0,NaN,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,per day,2.5,75.0,NaN,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,per day,2.5,75.0,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,per day,1.5,45.0,NaN,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,per day,2.0,60.0,NaN,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,per day,2.0,60.0,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,per month,40.0,40.0,OTT-Lite,1


In [44]:
run_sql_file(r'C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\05_usage_month.sql')

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\05_usage_month.sql


In [45]:
q("""select * from v_usage_month;""")

,customer_id,month,usage_days,total_mb,total_voice,roaming_days
0,1,2025-01,2,1222.5,60,0
1,1,2025-02,4,4496.3,114,0
2,1,2025-03,5,2871.0,50,0
3,1,2025-04,6,3815.2,109,0
4,2,2025-01,7,1870.5,225,1
...,...,...,...,...,...,...
117827,22000,2025-02,20,29489.8,1435,2
117828,22000,2025-03,20,33024.3,1489,0
117829,22000,2025-04,17,23536.9,1170,1
117830,22000,2025-05,23,39269.1,1521,1


In [46]:
run_sql_file(r'C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\06_customer_features.sql')

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\06_customer_features.sql


In [47]:
q("""SELECT COUNT(*) AS rows_,
COUNT(DISTINCT customer_id) AS subs,
SUM(usage_days = 0) AS no_usage,
SUM(recharge_count = 0) AS no_payment,
SUM(complaint_count = 0) AS never_complained,
ROUND(AVG(tenure_days), 1) AS tenure,
ROUND(AVG(actual_arpu_6m), 2) AS actual_arpu,
ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features;""")

,rows_,subs,no_usage,no_payment,never_complained,tenure,actual_arpu,churn
0,22000,22000,4,3711,6631,697.6,344.46,23.75


In [48]:
q("""SELECT c.customer_id,                      -- one row per sub
         NTILE(4) OVER (ORDER BY             -- 1 = longest
           COALESCE(u.recency_days, 999)     -- silence
           DESC) AS r_score,                 -- the R score
         NTILE(4) OVER (ORDER BY             -- 4 = pays most
           r.recharge_count) AS f_score,     -- often
         NTILE(4) OVER (ORDER BY             -- 4 = pays the
           r.recharge_total_6m) AS m_score   -- most money
  FROM   customers c                         -- all 22,000
  JOIN   v_usage_customer   u ON u.customer_id = c.customer_id -- Fix 2
  JOIN   v_revenue_customer r ON r.customer_id = c.customer_id ;-- Fix 1""")

,customer_id,r_score,f_score,m_score
0,266,1,1,1
1,10684,1,1,1
2,13508,1,3,3
3,19111,1,3,3
4,11341,1,1,1
...,...,...,...,...
21995,18304,4,4,3
21996,19504,4,4,3
21997,19685,4,4,3
21998,21196,4,4,3


In [49]:
run_sql_file(r'C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\06_customer_features.sql')

Run: C:\Users\muhsi\OneDrive\Desktop\SEMESTER_3\SEMESTER_3\Airtel_telecom_analytics\sql\practice\06_customer_features.sql


In [50]:
q("""select * from customer_features;""")

,customer_id,activation_date,tenure_days,tenure_years,observed_days,circle_name,region,plan_name,plan_type,price_inr,...,actual_arpu_6m,stated_arpu,arpu_gap,complaint_count,unresolved_count,resolved_count,avg_resolution_days,r_score,f_score,m_score
0,9,2022-09-04,1030,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,3,0,3,9.33,2,3,3
1,31,2024-02-10,506,1,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,3,2,3
2,151,2022-07-22,1074,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,349.00,374,25.00,4,1,3,5.33,2,3,3
3,186,2022-01-04,1273,3,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,4,2,3
4,207,2024-12-14,198,0,91,Delhi,North,Prepaid Value 209,Prepaid,209,...,69.67,224,154.33,3,0,3,5.33,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,21805,2022-09-03,1031,2,180,North East,East,Prepaid Max 479,Prepaid,479,...,159.67,257,97.33,2,0,2,10.00,2,2,2
21996,21831,2022-12-18,925,2,180,North East,East,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,2,2,3
21997,21906,2024-05-16,410,1,180,North East,East,Prepaid Entertainment 399,Prepaid,399,...,332.50,428,95.50,0,0,0,NaN,2,3,3
21998,21917,2024-02-25,491,1,180,North East,East,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,1,0,1,1.00,4,3,3


In [52]:
q("""select name, tbl_name from sqlite_master where type = 'index';""")

,name,tbl_name
0,ix_u_cust,usage_daily
1,ix_u_date,usage_daily
2,ix_r_cust,recharges
3,ix_c_cust,complaints
4,ix_cust_circle,customers
5,ix_cust_plan,customers
6,ix_cf_churn,customer_features
7,ix_cf_ab,customer_features
